# TCC Experimento 4: Treinamento do Detector de Placas Customizado (YOLOv8)

**Objetivo**: Treinar um modelo YOLOv8 Nano customizado para detecção de placas utilizando o dataset **UFPR-ALPR**. Para garantir a integridade metodológica do TCC, as imagens de treinamento são compostas por recortes (crops) dos veículos, e as caixas delimitadoras das placas são normalizadas em relação a esses recortes. Isso alinha perfeitamente a fase de treinamento com a de inferência (pipeline de duas etapas).

---
## Fluxo de Execução:
1. **Configuração e Ambiente**: Importações e verificação de hardware.
2. **Conversão do Dataset**: Leitura das anotações originais, recorte do veículo, normalização das coordenadas da placa no espaço do recorte e geração da estrutura do YOLO (`dataset_yolo/`).
3. **Treinamento**: Execução do treinamento da YOLOv8n no CPU/GPU.
4. **Validação**: Teste visual da detecção de placa em um crop de amostra.
5. **Salvamento do Modelo**: Cópia dos melhores pesos gerados (`best.pt`) para a raiz do projeto como `yolov8n-plate.pt` para integração imediata.

## 1. Configuração e Importações

In [1]:
import cv2
import os
import shutil
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO, settings

BASE_DIR = Path.cwd()
print(f"Diretório Base: {BASE_DIR}")
print(f"CUDA Disponível para Treino: {torch.cuda.is_available()}")

# Garante que as configurações globais do Ultralytics apontem para a pasta local deste projeto
settings.update({
    "datasets_dir": str(BASE_DIR),
    "weights_dir": str(BASE_DIR / "weights"),
    "runs_dir": str(BASE_DIR / "runs")
})
print("Configurações do Ultralytics atualizadas para o diretório base do projeto.")

Diretório Base: /home/nicman/Desktop/TCC1
CUDA Disponível para Treino: False
Configurações do Ultralytics atualizadas para o diretório base do projeto.


## 2. Preparação do Dataset (Crops de Veículos e Conversão YOLO)

Abaixo definimos as funções auxiliares para parsear as anotações do UFPR-ALPR, gerar os recortes de veículo e exportar no formato YOLOv8.

In [2]:
def parse_annotation(txt_path: Path) -> dict:
    """
    Lê um arquivo .txt do UFPR-ALPR e retorna um dicionário com:
        - vehicle_bbox : list  — [x1, y1, x2, y2]
        - plate_bbox   : list  — [x1, y1, x2, y2]
    """
    try:
        data = {}
        with open(txt_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            line = line.strip()
            if line.startswith('position_vehicle:'):
                parts = line.split(':')[1].strip().split()
                x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                data['vehicle_bbox'] = [x, y, x + w, y + h]
            elif line.startswith('corners:'):
                corners_str = line.split(':')[1].strip().split()
                corners = []
                for c in corners_str:
                    cx, cy = c.split(',')
                    corners.append([int(cx), int(cy)])
                xs = [p[0] for p in corners]
                ys = [p[1] for p in corners]
                data['plate_bbox'] = [min(xs), min(ys), max(xs), max(ys)]
        return data if 'plate_bbox' in data and 'vehicle_bbox' in data else None
    except Exception as e:
        print(f"Erro ao parsear {txt_path}: {e}")
        return None

def prepare_split(dataset_root: Path, output_root: Path, split_in: str, split_out: str, limit_tracks: int = None):
    src_split_path = dataset_root / split_in
    if not src_split_path.exists():
        print(f"Diretório {src_split_path} não encontrado. Pulando split '{split_out}'.")
        return

    img_out_dir = output_root / "images" / split_out
    lbl_out_dir = output_root / "labels" / split_out
    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    track_dirs = sorted([d for d in src_split_path.iterdir() if d.is_dir()])
    if limit_tracks:
        track_dirs = track_dirs[:limit_tracks]
        print(f"Processando subconjunto limitado a {limit_tracks} tracks...")

    processed_count = 0
    for track_dir in tqdm(track_dirs, desc=f"Convertendo {split_out}"):
        for img_path in sorted(track_dir.glob('*.png')):
            txt_path = img_path.with_suffix('.txt')
            if not txt_path.exists():
                continue
            annotation = parse_annotation(txt_path)
            if not annotation:
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            h_img, w_img = img.shape[:2]
            vx1, vy1, vx2, vy2 = annotation['vehicle_bbox']
            vx1, vy1 = max(0, vx1), max(0, vy1)
            vx2, vy2 = min(w_img, vx2), min(h_img, vy2)
            vw, vh = vx2 - vx1, vy2 - vy1
            if vw <= 0 or vh <= 0:
                continue

            # Crop do veículo
            vehicle_crop = img[vy1:vy2, vx1:vx2]

            # Coordenadas da placa no crop do veículo
            px1, py1, px2, py2 = annotation['plate_bbox']
            cpx1 = max(0, px1 - vx1)
            cpy1 = max(0, py1 - vy1)
            cpx2 = min(vw, px2 - vx1)
            cpy2 = min(vh, py2 - vy1)
            cpw = cpx2 - cpx1
            cph = cpy2 - cpy1
            if cpw <= 0 or cph <= 0:
                continue

            # YOLO format: class x_center y_center width height
            x_center = (cpx1 + cpw / 2.0) / vw
            y_center = (cpy1 + cph / 2.0) / vh
            norm_w = cpw / vw
            norm_h = cph / vh

            unique_name = f"{track_dir.name}_{img_path.stem}"
            cv2.imwrite(str(img_out_dir / f"{unique_name}.png"), vehicle_crop)
            with open(lbl_out_dir / f"{unique_name}.txt", 'w') as f:
                f.write(f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")
            processed_count += 1
    print(f"Split '{split_out}' concluído: {processed_count} crops gerados.")

### Executar a Conversão do Dataset

Selecione se deseja processar o dataset completo (1800 imagens de treino, 900 validação) ou apenas um subconjunto de teste rápido.

In [3]:
DATASET_ROOT = BASE_DIR / "UFPR-ALPR dataset"
OUTPUT_ROOT  = BASE_DIR / "dataset_yolo"

# Se True, processa apenas 2 tracks por split para testar se tudo compila rapidamente.
# Mude para False para processar o dataset completo.
SMOKE_TEST = True
LIMIT_TRACKS = 2 if SMOKE_TEST else None

print(f"Processando dados. SMOKE_TEST: {SMOKE_TEST}")
prepare_split(DATASET_ROOT, OUTPUT_ROOT, "training", "train", LIMIT_TRACKS)
prepare_split(DATASET_ROOT, OUTPUT_ROOT, "validation", "val", LIMIT_TRACKS)
prepare_split(DATASET_ROOT, OUTPUT_ROOT, "testing", "test", LIMIT_TRACKS)

# Escreve dataset.yaml
yaml_data = {
    "path": str(OUTPUT_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "license_plate"}
}
with open(OUTPUT_ROOT / "dataset.yaml", 'w', encoding='utf-8') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)
print("Arquivo dataset.yaml criado com sucesso!")

Processando dados. SMOKE_TEST: True
Processando subconjunto limitado a 2 tracks...


Convertendo train: 100%|██████████| 2/2 [00:02<00:00,  1.10s/it]


Split 'train' concluído: 60 crops gerados.
Processando subconjunto limitado a 2 tracks...


Convertendo val: 100%|██████████| 2/2 [00:02<00:00,  1.03s/it]


Split 'val' concluído: 60 crops gerados.
Processando subconjunto limitado a 2 tracks...


Convertendo test: 100%|██████████| 2/2 [00:02<00:00,  1.01s/it]

Split 'test' concluído: 60 crops gerados.
Arquivo dataset.yaml criado com sucesso!


## 3. Treinamento da YOLOv8 no CPU/GPU

In [4]:
device_type = "cuda" if torch.cuda.is_available() else "cpu"
epochs = 1 if SMOKE_TEST else 10
batch_size = 16
imgsz = 320

print(f"Carregando YOLOv8n base...")
model = YOLO("yolov8n.pt")

print(f"Iniciando treinamento ({epochs} épocas no {device_type})...")
model.train(
    data=str((OUTPUT_ROOT / "dataset.yaml").resolve()),
    epochs=epochs,
    batch=batch_size,
    imgsz=imgsz,
    device=device_type,
    project=str(BASE_DIR / "runs" / "detect"),
    name="train_plate",
    verbose=True
)
print("Treinamento finalizado!")

Carregando YOLOv8n base...
Iniciando treinamento (1 épocas no cpu)...
New https://pypi.org/project/ultralytics/8.4.87 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.53 🚀 Python-3.12.3 torch-2.12.0+cu130 CPU (AMD Ryzen 5 5500U with Radeon Graphics)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/nicman/Desktop/TCC1/dataset_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4,

## 4. Teste Visual das Detecções

Carregamos o modelo recém-treinado e rodamos em uma imagem do split de validação para visualizar o resultado.

In [5]:
trained_model_path = Path("runs/detect/train_plate/weights/best.pt")
if trained_model_path.exists():
    model_val = YOLO(trained_model_path)
    
    # Pega um crop da pasta de validação
    val_images = list((OUTPUT_ROOT / "images" / "val").glob("*.png"))
    if val_images:
        sample_img_path = val_images[0]
        img = cv2.imread(str(sample_img_path))
        results = model_val(img, verbose=False)[0]
        
        # Desenha caixas
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, f"placa {float(box.conf):.2f}", (x1, y1 - 5), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        plt.figure(figsize=(8, 6))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title("Detecção de Placa com o Modelo Customizado")
        plt.show()
    else:
        print("Nenhuma imagem de validação encontrada para visualização.")
else:
    print("Modelo treinado não encontrado em runs/detect/train_plate/weights/best.pt.")

<Figure size 800x600 with 1 Axes>

## 5. Salvar e Integrar o Modelo no Projeto

Copia os pesos treinados (`best.pt`) para o diretório raiz do projeto com o nome `yolov8n-plate.pt`. Isso substitui o modelo de fallback anterior pelo seu modelo de TCC de origem conhecida!

In [6]:
best_weights = Path("runs/detect/train_plate/weights/best.pt")
dest_weights = BASE_DIR / "yolov8n-plate.pt"

if best_weights.exists():
    shutil.copy(best_weights, dest_weights)
    print(f"Sucesso! Modelo salvo na raiz do seu projeto como:")
    print(f"  {dest_weights.resolve()}")
    print("\nO pipeline principal agora usará este modelo automaticamente!")
else:
    print("Erro: Pesos best.pt não encontrados para copiar.")

Sucesso! Modelo salvo na raiz do seu projeto como:
  /home/nicman/Desktop/TCC1/yolov8n-plate.pt

O pipeline principal agora usará este modelo automaticamente!
